# Thí nghiệm CNN-BiLSTM

Notebook này chạy biến thể CNN-BiLSTM từ `train_cnn_bilstm.py`. Mô hình giữ nguyên pipeline dữ liệu của CNN-LSTM chính, bổ sung Spatial Dropout, Batch Normalization và LSTM hai chiều. Ngoài ngưỡng mặc định 0,5, threshold được chọn trên validation để ưu tiên Attack Recall nhưng vẫn giữ Normal Recall tối thiểu.

## 1. Kiểm tra môi trường và đường dẫn

In [1]:
from pathlib import Path

print('Working directory:', Path.cwd())
required_files = [
    'train_cnn_bilstm.py',
    '../../cnn_lstm/CNN_LSTM.py',
    '../../SQLInjection_XSS_MixDataset.1.0.0.csv',
    '../../csic_database.csv',
    '../../obfuscation_dataset_full.xlsx',
]
for name in required_files:
    assert Path(name).exists(), f'Thiếu file: {name}'
print('Đã tìm thấy đầy đủ code và dataset.')

Working directory: c:\Users\Admin\OneDrive\Desktop\NCKH_sp\experiments\cnn_bilstm
Đã tìm thấy đầy đủ code và dataset.


## 2. Xem kiến trúc mà không huấn luyện

Luồng chính: `Embedding -> SpatialDropout -> CNN+BN -> Pool -> CNN+BN -> Pool -> Bidirectional LSTM -> Dense+BN -> Dropout -> Sigmoid`.

In [2]:
import importlib.util
import sys

module_path = Path('train_cnn_bilstm.py').resolve()
spec = importlib.util.spec_from_file_location('cnn_bilstm_experiment', module_path)
experiment = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = experiment
spec.loader.exec_module(experiment)

demo_model = experiment.build_improved_model(
    vocab_size=191,
    max_len=1024,
    embedding_dim=64,
)
demo_model.summary()
print('Tổng số tham số:', f'{demo_model.count_params():,}')

Model: "Improved_CNN_BiLSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_spatial_dropout       │ (None, 1024, 64)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_conv (Conv1D)           │ (None, 1024, 128)      │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_bn (BatchNormalization) │ (None, 1024, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_relu (Activation)       │ (None, 1024, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_conv (Conv1D)           │ (None, 256, 128)       │        81,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_bn (BatchNormalization) │ (None, 256, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_relu (Activation)       │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_context (Bidirectional)  │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_bn (BatchNormalization)   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_relu (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,073 (887.00 KB)

 Trainable params: 226,433 (884.50 KB)

 Non-trainable params: 640 (2.50 KB)

Tổng số tham số: 227,073


## 3. Smoke test tùy chọn

Cell này chỉ kiểm tra pipeline trên tập nhỏ. Không dùng kết quả smoke test trong báo cáo khoa học.

In [3]:
%run train_cnn_bilstm.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_bilstm_smoke

=== IMPROVED EXPERIMENT DATA ===
Train           : 2,160
Val             : 240
Test            : 600
Obfuscated test : 1,000
Vocabulary size : 111
Class weights   : {0: 1.4210526315789473, 1: 0.7714285714285715}


Model: "Improved_CNN_BiLSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │         7,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_spatial_dropout       │ (None, 1024, 64)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_conv (Conv1D)           │ (None, 1024, 128)      │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_bn (BatchNormalization) │ (None, 1024, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_relu (Activation)       │ (None, 1024, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_conv (Conv1D)           │ (None, 256, 128)       │        81,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_bn (BatchNormalization) │ (None, 256, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_relu (Activation)       │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_context (Bidirectional)  │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_bn (BatchNormalization)   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_relu (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 221,953 (867.00 KB)

 Trainable params: 221,313 (864.50 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6783 - auc: 0.7313 - loss: 0.6128 - precision: 0.7897 - recall: 0.6789
Epoch 1: val_loss improved from None to 0.67742, saving model to artifacts_bilstm_smoke\best_improved_cnn_bilstm.keras

Epoch 1: finished saving model to artifacts_bilstm_smoke\best_improved_cnn_bilstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.7639 - auc: 0.8511 - loss: 0.4839 - precision: 0.8666 - recall: 0.7514 - val_accuracy: 0.6500 - val_auc: 0.8706 - val_loss: 0.6774 - val_precision: 0.6500 - val_recall: 1.0000
Epoch 2/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8685 - auc: 0.9490 - loss: 0.2998 - precision: 0.9473 - recall: 0.8422
Epoch 2: val_loss improved from 0.67742 to 0.66610, saving model to artifacts_bilstm_smoke\best_improved_cnn_bilstm.keras

Epoch 2: finished saving model to artifacts_bilstm_smoke\best_improved_cnn_bilstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.8722 - auc: 0.9526 - loss: 0.2877

## 4. Huấn luyện đầy đủ

Mô hình dùng cùng split 72/8/20, tokenizer train-only, `MAX_LEN=1024` và obfuscation test riêng như mô hình chính. Cell này ghi đè artifact CNN-BiLSTM hiện tại.

In [6]:
%run train_cnn_bilstm.py --epochs 30 --batch-size 128 --min-normal-recall 0.99 --output-dir artifacts

=== IMPROVED EXPERIMENT DATA ===
Train           : 127,638
Val             : 14,183
Test            : 35,456
Obfuscated test : 150,000
Vocabulary size : 191
Class weights   : {0: 1.3997543482552146, 1: 0.7778536169175453}


Model: "Improved_CNN_BiLSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_spatial_dropout       │ (None, 1024, 64)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_conv (Conv1D)           │ (None, 1024, 128)      │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_bn (BatchNormalization) │ (None, 1024, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3_relu (Activation)       │ (None, 1024, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_conv (Conv1D)           │ (None, 256, 128)       │        81,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_bn (BatchNormalization) │ (None, 256, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5_relu (Activation)       │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_context (Bidirectional)  │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_bn (BatchNormalization)   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_relu (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,073 (887.00 KB)

 Trainable params: 226,433 (884.50 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/30
 11/998 ━━━━━━━━━━━━━━━━━━━━ 17:46 1s/step - accuracy: 0.6514 - auc: 0.6809 - loss: 0.6650 - precision: 0.7694 - recall: 0.6483

KeyboardInterrupt: 

## 5. Đọc kết quả CNN-BiLSTM

Nếu chưa muốn train lại, có thể chạy trực tiếp cell này để đọc `artifacts/metadata_and_results.json` đã lưu.

In [ ]:
import json
import pandas as pd

def load_json(path):
    with open(path, encoding='utf-8') as file:
        return json.load(file)

bilstm = load_json('artifacts/metadata_and_results.json')
model_info = bilstm['improved_model']
evaluation = bilstm['evaluation']

rows = []
for dataset, result_key in [
    ('Test @ 0.5', 'test_default_0_5'),
    ('Test @ tuned', 'test_tuned'),
    ('Obfuscation @ 0.5', 'obfuscated_default_0_5'),
    ('Obfuscation @ tuned', 'obfuscated_tuned'),
]:
    result = evaluation[result_key]
    cm = result['confusion_matrix']
    total = sum(sum(row) for row in cm)
    accuracy = (cm[0][0] + cm[1][1]) / total if total and sum(cm[0]) else None
    rows.append({
        'Tập đánh giá': dataset,
        'Threshold': result['threshold'],
        'Accuracy': accuracy,
        'Attack Recall': result['attack_recall'],
        'Attack F1': result['attack_f1'],
        'False Positive': cm[0][1],
        'False Negative': cm[1][0],
    })

print('Threshold được chọn:', model_info['selected_threshold'])
pd.DataFrame(rows)

## 6. Kiểm tra quá trình chọn threshold

Threshold chỉ được chọn bằng validation. Không được dùng test hoặc obfuscation test để chọn threshold.

In [ ]:
threshold_table = pd.DataFrame(bilstm['threshold_search_val'])
threshold_table[[
    'threshold',
    'normal_recall',
    'attack_recall',
    'attack_f1',
]].sort_values('threshold')

## 7. So sánh với CNN-only và CNN-LSTM chính

Bảng dưới dùng ngưỡng 0,5 cho cả ba mô hình để tránh tạo lợi thế riêng cho CNN-BiLSTM.

In [ ]:
cnn = load_json('../../cnn_only/artifacts/metadata_and_results.json')
hybrid = load_json('../../cnn_lstm/artifacts/metadata_and_results.json')

cnn_test = cnn['evaluation']['normal_test']
cnn_obfu = cnn['evaluation']['obfuscated_test']
hybrid_test = hybrid['evaluation']['test']
hybrid_obfu = hybrid['evaluation']['obfuscated_test']
bilstm_test = evaluation['test_default_0_5']
bilstm_obfu = evaluation['obfuscated_default_0_5']
bilstm_cm = bilstm_test['confusion_matrix']
bilstm_accuracy = (bilstm_cm[0][0] + bilstm_cm[1][1]) / sum(sum(row) for row in bilstm_cm)

comparison = [
    {
        'Mô hình': 'CNN-only',
        'Test Accuracy': cnn_test['accuracy'],
        'Attack Recall': cnn_test['classification_report']['Attack (1)']['recall'],
        'Attack F1': cnn_test['classification_report']['Attack (1)']['f1-score'],
        'Obfuscation Recall': cnn_obfu['classification_report']['1']['recall'],
        'Obfuscation Missed': cnn_obfu['confusion_matrix'][1][0],
    },
    {
        'Mô hình': 'CNN-LSTM',
        'Test Accuracy': hybrid_test['accuracy'],
        'Attack Recall': hybrid_test['classification_report']['Attack (1)']['recall'],
        'Attack F1': hybrid_test['classification_report']['Attack (1)']['f1-score'],
        'Obfuscation Recall': hybrid_obfu['classification_report']['1']['recall'],
        'Obfuscation Missed': hybrid_obfu['confusion_matrix'][1][0],
    },
    {
        'Mô hình': 'CNN-BiLSTM',
        'Test Accuracy': bilstm_accuracy,
        'Attack Recall': bilstm_test['attack_recall'],
        'Attack F1': bilstm_test['attack_f1'],
        'Obfuscation Recall': bilstm_obfu['attack_recall'],
        'Obfuscation Missed': bilstm_obfu['confusion_matrix'][1][0],
    },
]

pd.DataFrame(comparison)

## Cách diễn giải

BiLSTM chỉ được xem là cải tiến nếu lợi ích trên test và obfuscation đủ lớn so với chi phí tham số/thời gian. Không kết luận dựa riêng vào Accuracy. Cần xem Attack Recall, false negative, benign false positive và độ ổn định qua nhiều seed.